# Bio-VQA — Colab notebook

Runs VQA models on figures extracted from PMC Open Access archives.
All models are free and open-source.

**Before running: `Runtime → Change runtime type → T4 GPU`**

| Model | VRAM | Notes |
|---|---|---|
| `qwen2.5-vl-3b` | 8 GB | Best free T4 option |
| `paligemma-3b` | 7 GB | Good on scientific figures. Needs licence on HF. |
| `internvl2-2b` | 5 GB | Smallest, reliable fallback |
| `llava-1.5-7b` | 14 GB | Fits T4 (barely) |
| `florence-2-large` | 4 GB | Captions only — not real VQA |


In [ ]:
# Step 1 — get the code
import os
REPO = "https://github.com/YOUR_USERNAME/bio-vqa.git"  # <- update this

if not os.path.exists('/content/bio-vqa'):
    !git clone {REPO} /content/bio-vqa
else:
    !git -C /content/bio-vqa pull

!pip install -q -r /content/bio-vqa/requirements.txt
import sys; sys.path.insert(0, '/content/bio-vqa')
print('done')

In [ ]:
# Step 2 — upload your PMC archive
from google.colab import files
uploaded = files.upload()
archive_path = list(uploaded.keys())[0]
print('archive:', archive_path)

In [ ]:
# Step 3 — preview figures
import tempfile
import matplotlib.pyplot as plt
from PIL import Image
from bio_vqa import parse_archive, list_models

list_models()

_tmp = tempfile.mkdtemp()
figures = parse_archive(archive_path, _tmp)
print(f'\nFound {len(figures)} figure(s)')
for fig in figures:
    print(f'  {fig.figure_id}: {fig.label} — {fig.caption[:100]}...')

cols = min(len(figures), 3)
rows = (len(figures) + cols - 1) // cols
_, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
ax_list = [axes] if len(figures) == 1 else list(axes.flat if rows > 1 else axes)
for fig_obj, ax in zip(figures, ax_list):
    ax.imshow(Image.open(fig_obj.image_path))
    ax.set_title(fig_obj.label, fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Step 4 — run the pipeline
from bio_vqa import run_pipeline

MODEL     = ['qwen2.5-vl-3b']  # change to try other models
OUT_DIR   = 'vqa_output'
MAX_FIGS  = None  # set e.g. 1 for a quick test

results = run_pipeline(
    archive_path=archive_path,
    model_keys=MODEL,
    output_dir=OUT_DIR,
    max_figures=MAX_FIGS,
)
print('done')

In [ ]:
# Step 5 — browse results
import json
from collections import defaultdict
from PIL import Image
import matplotlib.pyplot as plt

with open(f'{OUT_DIR}/results.json') as f:
    data = json.load(f)

fig_map = {f['figure_id']: f for f in data['figures']}
by_fig = defaultdict(list)
for r in data['results']:
    by_fig[r['figure_id']].append(r)

for fig_id, qa in by_fig.items():
    meta = fig_map[fig_id]
    plt.figure(figsize=(10, 4))
    plt.imshow(Image.open(meta['image_path']))
    plt.title(meta['label']); plt.axis('off'); plt.show()
    print('Caption:', meta['caption'][:200])
    for r in qa:
        print(f"\n[{r['question_type']}]")
        print('Q:', r['question'][:80])
        print('A:', r['answer'])
        print(f"score: {r['composite_score']}  latency: {r['latency_s']}s")
    print('='*70)

In [ ]:
# Step 6 — download results
import shutil
from google.colab import files
shutil.make_archive('vqa_output', 'zip', OUT_DIR)
files.download('vqa_output.zip')

## Troubleshooting

**Out of memory** → try `internvl2-2b` (5 GB) or `paligemma-3b` (7 GB)

**Same answer every question** → that's Florence-2 (captioning model, not VQA) — switch to qwen

**PaliGemma 403** → accept the licence at hf.co/google/paligemma-3b-mix-448 then set `HF_TOKEN`

**Re-downloading weights every session** → mount Drive and add before Step 4:
```python
from google.colab import drive
drive.mount('/content/drive')
import os; os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
```
